# 🔧 Module 3 — Class 5 Homework: Pipelines

**Topic:** Pipelines — ColumnTransformer, Pipeline, joblib

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [1]:
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print('Loaded:', df.shape)


Loaded: (7043, 21)


In [2]:
# Cell 0 explanation:
# Import pandas, numpy, matplotlib and suppress warnings
# Load the Telco Customer Churn dataset from GitHub and clean the TotalCharges column

### Cell 1 — set up the target and features

In [3]:
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = ['gender', 'Contract', 'PaymentMethod']
X = df[num_features + cat_features]
y = df['Churn_bin']
print('X:', X.shape, 'y:', y.shape)

X: (7043, 6) y: (7043,)


In [4]:
# Cell 1 explanation:
# Map Churn to 0/1, then define the numeric and categorical features to build X and y

### Cell 2 — split into train and test

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5634, 6), Test: (1409, 6)


In [6]:
# Cell 2 explanation:
# Split the data into training and test sets

### Cell 3 — build the numeric preprocessor

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
num_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [8]:
# Cell 3 explanation:
# Build a pipeline for numeric features that fills missing values with the median and then scales them with StandardScaler

### Cell 4 — build the categorical preprocessor

In [9]:
from sklearn.preprocessing import OneHotEncoder
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
cat_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [10]:
# Cell 4 explanation:
# Build a pipeline for categorical features that fills missing values with the most frequent value and then one-hot encodes them

### Cell 5 — combine both with ColumnTransformer

In [11]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Contract', 'PaymentMethod'])])

In [12]:
# Cell 5 explanation:
# Combine the numeric and categorical pipelines into one preprocessor using ColumnTransformer

### Cell 6 — full pipeline: preprocessor + model

In [13]:
from sklearn.linear_model import LogisticRegression
full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])
full_pipe

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Contract',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [14]:
# Cell 6 explanation:
# Build the full pipeline by chaining the preprocessor with a LogisticRegression model

### Cell 7 — fit on training data

In [15]:
full_pipe.fit(X_train, y_train)
print('✅ Pipeline trained')

✅ Pipeline trained


In [16]:
# Cell 7 explanation:
# Fit the entire pipeline on the training data in one step

### Cell 8 — evaluate on test data

In [17]:
from sklearn.metrics import accuracy_score
y_pred = full_pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}')

Test accuracy: 0.7963


In [18]:
# Cell 8 explanation:
# Predict on the test data and calculate the model's accuracy

### Cell 9 — save the trained pipeline

In [19]:
import joblib
joblib.dump(full_pipe, 'churn_pipeline.joblib')
loaded = joblib.load('churn_pipeline.joblib')
print('Saved and loaded.')
print('Test accuracy from loaded model:', round(loaded.score(X_test, y_test), 4))

Saved and loaded.
Test accuracy from loaded model: 0.7963


In [20]:
# Cell 9 explanation:
# Save the trained pipeline to disk with joblib, then load it back and confirm it gives the same test accuracy

---

## 📤 Submit

Comment every cell. Push to `module-3/class_5/submissions/<YourName>/`.